In [0]:
import importlib
import utils.helpers

importlib.reload(utils.helpers)

In [0]:
%restart_python

In [0]:
base_path = f"/Volumes/bronze_dev/nppes_npi_registry/nppes_raw/"
zip_path = base_path + "nppes.zip"
extract_path = base_path + "unzipped/"

# Step 1: Fetch the data

## Download zip file directly to volume
Zip file is successfully downloaded to `dbfs:/Volumes/bronze_dev/nppes_npi_registry/nppes_raw/nppes.zip`

In [0]:
dbutils.fs.mkdirs(base_path)

dbutils.fs.cp(
    "https://download.cms.gov/nppes/NPPES_Data_Dissemination_March_2026_V2.zip",
    zip_path
)

Databricks view

In [0]:
dbutils.fs.ls(base_path)

In [0]:
display(dbutils.fs.ls(base_path))

# Unzip the data in Python with `zipfile` library

In [0]:
print(f"{zip_path=}")
print(f"{extract_path=}")

In [0]:
import zipfile

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [0]:
display(dbutils.fs.ls(extract_path))

In [0]:
import pandas as pd
import numpy as np


columns_to_keep = [
	"NPI",
	"Entity Type Code",
	"Provider Organization Name (Legal Business Name)",
	"Provider Last Name (Legal Name)",
	"Provider First Name",
	"Provider Credential Text",
	"Provider Other Organization Name",
	"Provider Other Organization Name Type Code",
	"Provider Business Mailing Address City Name",
	"Provider Business Mailing Address State Name",
	"Provider Business Mailing Address Postal Code",
	"Provider Business Practice Location Address City Name",
	"Provider Business Practice Location Address State Name",
	"Provider Business Practice Location Address Postal Code",
	"NPI Deactivation Reason Code",
	"Provider Sex Code",
    "Healthcare Provider Taxonomy Code_1",
	"Healthcare Provider Primary Taxonomy Switch_1",
	"Healthcare Provider Taxonomy Code_2",
	"Healthcare Provider Primary Taxonomy Switch_2",
	"Healthcare Provider Taxonomy Code_3",
	"Healthcare Provider Primary Taxonomy Switch_3",
	"Healthcare Provider Taxonomy Code_4",
    "Healthcare Provider Primary Taxonomy Switch_4",
	"Healthcare Provider Taxonomy Code_5",
	"Healthcare Provider Primary Taxonomy Switch_5",
	"Healthcare Provider Taxonomy Code_6",
	"Healthcare Provider Primary Taxonomy Switch_6",
	"Healthcare Provider Taxonomy Code_7",
	"Healthcare Provider Primary Taxonomy Switch_7",
	"Healthcare Provider Taxonomy Code_8",
	"Healthcare Provider Primary Taxonomy Switch_8",
	"Healthcare Provider Taxonomy Code_9",
	"Healthcare Provider Primary Taxonomy Switch_9",
	"Healthcare Provider Taxonomy Code_10",
	"Healthcare Provider Primary Taxonomy Switch_10",
	"Healthcare Provider Taxonomy Code_11",
	"Healthcare Provider Primary Taxonomy Switch_11",
	"Healthcare Provider Taxonomy Code_12",
	"Healthcare Provider Primary Taxonomy Switch_12",
	"Healthcare Provider Taxonomy Code_13",
	"Healthcare Provider Primary Taxonomy Switch_13",
	"Healthcare Provider Taxonomy Code_14",
	"Healthcare Provider Primary Taxonomy Switch_14",
	"Healthcare Provider Taxonomy Code_15",
	"Healthcare Provider Primary Taxonomy Switch_15",
	"Is Sole Proprietor",
	"Is Organization Subpart"
]

df = pd.read_csv(
    f"{extract_path}/npidata_pfile_20050523-20260308.csv",
    dtype=str,
    usecols=columns_to_keep
)

In [0]:
spark_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)  # equivalent to dtype=str in pandas
    .csv(f"{extract_path}/npidata_pfile_20050523-20260308.csv")
    .select(columns_to_keep)
)

In [0]:
display(spark_df)

In [0]:
from pyspark.sql import functions as F

# Build lists of taxonomy + switch columns
taxonomy_cols = [f"Healthcare Provider Taxonomy Code_{i}" for i in range(1, 16)]
switch_cols = [f"Healthcare Provider Primary Taxonomy Switch_{i}" for i in range(1, 16)]

# Build a list of conditional expressions, one per (taxonomy, switch) pair.
#
# For each pair:
# - If the "Primary Taxonomy Switch" column == 'Y', return the corresponding taxonomy code
# - Otherwise return NULL
#
# This produces a list like:
# [
#   when(col("Switch_1") == 'Y', col("Taxonomy_1")),
#   when(col("Switch_2") == 'Y', col("Taxonomy_2")),
#   ...
# ]
#
# Important:
# - Each expression evaluates independently (no sequential overwriting like pandas)
# - We'll later combine these using `coalesce`, which picks the first non-null value
#   → effectively: "first taxonomy where switch == 'Y'"

# Build conditional expressions
exprs = [
    F.when(F.col(switch) == "Y", F.col(taxonomy))
    for taxonomy, switch in zip(taxonomy_cols, switch_cols)
]
exprs

In [0]:

# Combine using coalesce (first non-null wins)
spark_df = spark_df.withColumn(
    "current_primary_taxonomy",
    F.coalesce(*exprs)
)

# Preview
display(spark_df.select(*taxonomy_cols, *switch_cols, "current_primary_taxonomy"))

In [0]:
# Count how many switch columns == 'Y' per row
spark_df = spark_df.withColumn(
    "primary_taxonomy_Y_count",
    # For each column: convert 'Y' → 1, everything else → 0, then sum across the row
    sum(F.when(F.col(c) == "Y", 1).otherwise(0) for c in switch_cols)
)

# Flag rows with multiple 'Y's
spark_df = spark_df.withColumn(
    "multiple_primary_flag",
    F.col("primary_taxonomy_Y_count") > 1
)

# Could do `if spark_df.filter(F.col("multiple_primary_flag")).limit(1).count() > 0:`
# But better to avoid .count() (which scans everything). This stops early:
if spark_df.filter(F.col("multiple_primary_flag")).take(1):
    print("Warning: There are providers with multiple primary taxonomies. Please review the flagged rows.")

    display(
        spark_df.filter(F.col("multiple_primary_flag"))
    )

In [0]:
# Drop unnecessary columns
cols_to_drop = taxonomy_cols + switch_cols + [
    "primary_taxonomy_Y_count",
    "multiple_primary_flag"
]

spark_df = spark_df.drop(*cols_to_drop)

In [0]:
df_prepped = utils.helpers.prep_bronze_df(spark_df)

In [0]:
df_prepped.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_dev.nppes_npi_registry.npi")